# 04. Chunk, Embedding, Retrieval 학습 흐름

검증 목적: 위에서 아래로 실행하며 튜토리얼 앱의 구조와 API 계약을 직접 확인합니다.

설명 셀과 코드 셀을 번갈아 두었습니다. 이 프로젝트만 열어도 읽고 바로 실행할 수 있습니다.

이 노트북에서 확인할 내용: `외부 모델 없이도 청크, 간단 벡터화, 유사도 검색의 원리를 작은 예제로 확인합니다.`
관련 장: 04 Chunk/Embedding, 05 Chroma

## 실행 전 준비

- 저장소 루트에서 Jupyter 커널을 시작합니다.
- 긴 서버를 백그라운드로 띄우지 않고, 가능한 한 TestClient와 파일 읽기로 확인합니다.
- 개인 `.env` 값, API 키, 로컬 DB 경로는 출력하지 않습니다.
- 이번 노트북에서는 튜토리얼 앱의 Chroma/Ollama 흐름을 보기 전에 검색 원리를 손으로 따라가 봅니다.

In [ ]:
# 공통 경로 셀
# 모든 노트북은 저장소 루트에서 실행한다고 가정합니다.
from pathlib import Path
PROJECT_ROOT = Path.cwd()
TEMPLATE_ROOT = PROJECT_ROOT / 'project_template'
print('PROJECT_ROOT:', PROJECT_ROOT.name)
print('TEMPLATE_ROOT exists:', TEMPLATE_ROOT.exists())
assert TEMPLATE_ROOT.exists(), 'project_template 폴더가 보여야 합니다.'

## 튜토리얼 앱 연결

이제 작은 실험으로 튜토리얼 앱의 어느 파일과 이어지는지 확인합니다. 코드가 길어 보여도 볼 것은 하나입니다. 출력이 예상과 다르면 바로 앞 셀부터 다시 확인하세요.

### 1. 샘플 문서 준비

이 셀에서는 `샘플 문서 준비` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [ ]:
documents = [
    '부산 해운대는 바다 관광과 산책로가 유명합니다.',
    '경주는 문화유산과 역사 여행지로 많이 찾습니다.',
    '제주는 자연 경관과 무장애 관광 코스가 중요합니다.',
]
for i, doc in enumerate(documents):
    print(i, doc)

### 2. 간단 청크 만들기

이 셀에서는 `간단 청크 만들기` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [ ]:
chunks = []
for doc_id, doc in enumerate(documents):
    for sentence in doc.split('.'):
        sentence = sentence.strip()
        if sentence:
            chunks.append({'doc_id': doc_id, 'text': sentence})
print(chunks)
assert len(chunks) == 3

### 3. 토큰 집합 벡터화

이 셀에서는 `토큰 집합 벡터화` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [ ]:
def tokenize(text: str) -> set[str]:
    return {token.strip('.,!?') for token in text.split() if token.strip('.,!?')}
for chunk in chunks:
    chunk['tokens'] = tokenize(chunk['text'])
print(chunks[0])

### 4. Jaccard 유사도

이 셀에서는 `Jaccard 유사도` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [ ]:
def score(query: str, chunk: dict) -> float:
    q = tokenize(query)
    c = chunk['tokens']
    return len(q & c) / max(1, len(q | c))
query = '부산 바다 관광'
ranked = sorted(chunks, key=lambda chunk: score(query, chunk), reverse=True)
for item in ranked:
    print(score(query, item), item['text'])
assert ranked[0]['doc_id'] == 0

### 5. project_template 검색 서비스 위치 확인

이 셀에서는 `project_template 검색 서비스 위치 확인` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [ ]:
from pathlib import Path
service_files = ['embedding_service.py', 'vector_store.py', 'retriever.py', 'text_splitter.py']
SERVICE_ROOT = Path.cwd() / 'project_template' / 'app' / 'services'
for name in service_files:
    path = SERVICE_ROOT / name
    print(name, path.exists())
    assert path.exists()

### 6. Chroma 런타임 제외 정책 확인

이 셀에서는 `Chroma 런타임 제외 정책 확인` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [ ]:
vector_dir = Path.cwd() / 'project_template' / 'data' / 'vector_store'
print('vector dir exists:', vector_dir.exists())
print('runtime files:', [p.name for p in vector_dir.iterdir()] if vector_dir.exists() else [])
assert vector_dir.exists()

## 정리

여기서는 최종 앱 전체가 아니라 이 장에서 확인해야 할 핵심 계약만 봤습니다. 같은 원리는 `project_template/app`, `project_template/frontend`, `project_template/data` 안의 실제 파일로 이어집니다.

In [ ]:
summary = {
    'notebook': 'completed',
    'next_step': '관련 chapter 문서를 읽고 같은 검증을 테스트로 반복합니다.',
}
print(summary)
assert summary['notebook'] == 'completed'